# Rabi calibration with QCal Copilot

Load a raw `.npy` Rabi trace, auto-fit a damped sine, hand the plot + fit numbers to
the Ising Calibration VLM, and emit a CUDA-Q script seeded with the recommended
pulse amplitude.

Inputs expected: a 1-D numpy array of readout populations vs drive amplitude (or time)
and an optional matching `x` array. Works offline — the local VLM is optional.

In [ ]:
import numpy as np

# Synthesize a Rabi sweep (replace with `np.load('your_trace.npy')`).
t = np.linspace(0, 500e-9, 201)              # seconds
y = 0.5 * np.exp(-t/300e-9) * np.sin(2*np.pi*10e6*t) + 0.5
np.save('rabi_trace.npy', y)
np.save('rabi_time.npy', t)

In [ ]:
from qcal.data import from_npy

payload = from_npy(
    'rabi_trace.npy',
    experiment_type='rabi',
    x_path='rabi_time.npy',
    x_unit='s',
)
payload.fit  # FitResult — rich-displays in Jupyter as a table

In [ ]:
payload.image  # the rendered plot the VLM will see

In [ ]:
from qcal.analyzer import analyze_payload

# backend='auto' uses NIM when NVIDIA_API_KEY is set, else the local HF weights.
result = analyze_payload(payload, backend='auto')
result  # _repr_markdown_ — shows experiment, issues, recommended params

In [ ]:
from qcal.codegen import generate_script

print(generate_script(result.parsed))